# Step 5 — Predictive Analyst (the fine-tune target ⭐)

Predicts the hardest investor questions from **three compounding signals** (P1 update):

1. **Segment/geo YoY swings** (structured signals) — the primary source of hard questions;
   questions like "Data Center rose 92% YoY — what is driving that?" are grounded in real
   defeatbeta-api breakdown data.
2. **Peer lags** — metrics where the company trails the peer set.
3. **RAG over the multimodal wiki** — actual analyst questions from past earnings calls,
   retrieved by the BM25-blend reranker across all doc types.

The analyst role is correctly classified (exec/operator paragraphs excluded — P1 fix).
`LLM_BACKEND=vllm` feeds the (fine-tuned) model for sharper phrasing; offline uses grounded
templates.  See `docs/finetuning.md`.

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root(); sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print(f"ticker={settings.ticker}  period={settings.period}")
print(f"data={'mock(real cached)' if settings.use_mock_data else 'live defeatbeta-api'}  "
      f"embeddings={settings.embedding_backend}  sentiment={settings.sentiment_backend}  llm={settings.llm_backend}")

ticker=NVDA  period=FY2026Q2
data=mock(real cached)  embeddings=hash  sentiment=lexicon  llm=mock


In [2]:
from ir_copilot.facts import build_fact_store
from ir_copilot.embeddings import get_embedder
from ir_copilot.vectorstore import WikiStore, WikiChunk
from ir_copilot.corpus import wiki_chunks_for, news_items, signals
from ir_copilot.agents.sentiment import analyze_sentiment
from ir_copilot.agents.competitor import compare
from ir_copilot.agents.predictive import predict_questions
from ir_copilot.llm import get_chat
import collections

# ── Facts
store = build_fact_store(settings.ticker, settings.period, use_mock=settings.use_mock_data)
peers = {p: build_fact_store(p, settings.period, use_mock=settings.use_mock_data)
         for p in settings.peers}
print(f"Facts: {len(store.facts)} ({settings.ticker})  Peers: {list(peers)}")

# ── Multimodal wiki (all six doc types)
tickers = [settings.ticker, *settings.peers]
all_chunks = wiki_chunks_for(tickers)
by_type = collections.Counter(c["doc_type"] for c in all_chunks)
wiki = WikiStore(get_embedder()); wiki.ensure_collection(recreate=True)
wiki.upsert([WikiChunk(chunk_id=str(i), **c) for i, c in enumerate(all_chunks)])
print(f"Wiki: {sum(by_type.values())} chunks  {dict(by_type)}")

# ── Sentiment + competitor + signals
snap = analyze_sentiment(settings.ticker, news_items(settings.ticker))
pc   = compare(store, peers)
sigs = signals(settings.ticker)
print(f"\nSentiment: net={snap.net_score:.2f}  Signals: {len(sigs)} segment/geo movers")
if sigs:
    print("  Top signals:", [(s['series'], f"{s['change_pct']:+.0f}%") for s in sigs[:3]])

Facts: 27 (NVDA)  Peers: ['AMD', 'TSLA']


Wiki: 177 chunks  {'transcript': 108, 'financial': 3, 'segment': 4, 'geo': 2, 'filing': 36, 'news': 24}

Sentiment: net=0.00  Signals: 6 segment/geo movers
  Top signals: [('Data Center Revenue', '+92%'), ('Compute and Networking revenue', '+88%'), ('Taiwan revenue', '+68%')]


In [3]:
# ── Predict questions
questions = predict_questions(store, snap, pc, wiki=wiki,
                              signals=sigs, chat=get_chat("analyst"))

print(f"\n{'='*65}")
print(f"  {len(questions)} predicted questions (ranked by difficulty)")
print(f"{'='*65}")
for i, q in enumerate(questions, 1):
    src_types = list({u.split('(')[1].split()[0] if '(' in u else 'wiki' for u in q.evidence if u})[0] if q.evidence else "signal"
    print(f"\n{i}. [{q.difficulty:.2f}] {q.text}")
    print(f"   ↳ why: {q.rationale}")
    print(f"   ↳ evidence type: {src_types} | url: {q.evidence[0][:70] if q.evidence else '–'}")

assert questions, "expected predicted questions"
assert any("segment" in q.rationale.lower() or "%" in q.text or "YoY" in q.text or "year-over-year" in q.text.lower()
           for q in questions), "expected at least one segment/YoY-driven question"
print("\nPredictive Analyst: segment signals + peer lags + real analyst Qs — verified.")


  8 predicted questions (ranked by difficulty)

1. [1.00] Data Center Revenue rose 92% year-over-year — what is driving that and how durable is it?
   ↳ why: Data Center Revenue revenue moved 92.4% YoY (segment breakdown). Precedent: "Thank you for taking my question. Could I just you wouldn't mind confirming if Q1 is the b..."
   ↳ evidence type: quarterly_revenue_by_breakdown | url: https://huggingface.co/datasets/defeatbeta/yahoo-finance-data (quarter

2. [0.99] Compute and Networking revenue rose 88% year-over-year — what is driving that and how durable is it?
   ↳ why: Compute and Networking revenue revenue moved 88.3% YoY (segment breakdown). Precedent: "We now offer three networking technologies. One is for scale-up, one is for scale-out, and..."
   ↳ evidence type: quarterly_revenue_by_breakdown | url: https://huggingface.co/datasets/defeatbeta/yahoo-finance-data (quarter

3. [0.89] Taiwan revenue rose 68% year-over-year — what is driving that and how durable is it?
   ↳ why: 

### The fine-tuning win (see `08_finetuning_unsloth.ipynb`)
With `LLM_BACKEND=mock` the question text is templated but grounded. With `LLM_BACKEND=vllm`
(MI300X), the fine-tuned LoRA adapter phrases the same grounded questions more naturally and
requires **zero few-shot examples** in the prompt (~60–70% fewer tokens).

**Next (Step 6):** draft the script/deck/Q&A and verify grounding.